### 现在进入 — 第五周第三天

## AutoGen Core

一些不太一样的东西。

这与底层的 Agent 框架无关。

你可以使用 AutoGen AgentChat，也可以使用其他框架；它是一个 Agent 交互框架。

从这个角度来看，它的定位与 LangGraph 类似。

### 核心原则

Autogen Core 将 Agent 的逻辑与消息的传递方式解耦。  
该框架提供通信基础设施以及 Agent 生命周期管理，而 Agent 则负责各自的具体工作。

通信基础设施被称为 **Runtime**（运行时）。

Runtime 有两种类型：**Standalone（单机）** 和 **Distributed（分布式）**。

今天我们使用单机运行时：**SingleThreadedAgentRuntime**，一个本地嵌入式 Agent 运行时实现。

明天我们将简要了解分布式运行时。


In [ ]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)


### 首先定义我们的消息对象（Message）

在我们的 Agent 框架中，消息可以是我们想要的任何结构。

In [ ]:
# 来一个简单的！

@dataclass
class Message:
    content: str


### 现在定义我们的 Agent

继承自 RoutedAgent。

每个 Agent 都有一个 **Agent ID**，由两个部分组成：  
`agent.id.type` 描述该 Agent 的类型  
`agent.id.key` 赋予其唯一标识符

任何使用 `@message_handler` 装饰器标记的方法都有机会接收消息。


In [ ]:
class SimpleAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("Simple")
#@function_tool 之前也有工具装饰器在2_openai\me\agenttools.py 中有
    @message_handler
    async def on_my_message(self, message: Message, ctx: MessageContext) -> Message:
        return Message(content=f"This is {self.id.type}-{self.id.key}. You said '{message.content}' and I disagree.")
        

### 好的，我们来创建一个 Standalone 运行时，并注册我们的 Agent 类型

In [ ]:

runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())

### 好了！启动运行时并发送一条消息

In [5]:
runtime.start()

In [ ]:
agent_id = AgentId("simple_agent", "default")
response = await runtime.send_message(Message("Well hi there!"), agent_id)
print(">>>", response.content)

In [7]:
await runtime.stop()
await runtime.close()

### 好了，现在来做点更有趣的事情

我们将使用一个 AgentChat Assistant！

In [8]:

class MyLLMAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("LLMAgent")
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent("LLMAgent", model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received message: {message.content}")
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        reply = response.chat_message.content
        print(f"{self.id.type} responded: {reply}")
        return Message(content=reply)
    


In [ ]:
from autogen_core import SingleThreadedAgentRuntime

runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())
await MyLLMAgent.register(runtime, "LLMAgent", lambda: MyLLMAgent())

In [ ]:
runtime.start()  # 在后台开始处理消息。
response = await runtime.send_message(Message("你好！"), AgentId("LLMAgent", "default"))
print(">>>", response.content)
response =  await runtime.send_message(Message(response.content), AgentId("simple_agent", "default"))
print(">>>", response.content)
response = await runtime.send_message(Message(response.content), AgentId("LLMAgent", "default"))

In [18]:
await runtime.stop()
await runtime.close()

### 好了，现在展示一下实际效果——让 3 个 Agent 互相交互！

In [19]:
from autogen_ext.models.ollama import OllamaChatCompletionClient


class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OllamaChatCompletionClient(model="llama3.2", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)

In [ ]:
JUDGE = "你正在评判一场石头剪刀布游戏。玩家们做出了以下选择：\n"

class RockPaperScissorsAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        instruction = "你正在玩石头剪刀布。只用一个词回复，从以下选项中选择：rock（石头）、paper（布）或 scissors（剪刀）。"
        message = Message(content=instruction)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message, inner_1)
        response2 = await self.send_message(message, inner_2)
        result = f"玩家 1: {response1.content}\n玩家 2: {response2.content}\n"
        judgement = f"{JUDGE}{result}谁赢了？"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + response.chat_message.content)


In [21]:
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, "player1", lambda: Player1Agent("player1"))
await Player2Agent.register(runtime, "player2", lambda: Player2Agent("player2"))
await RockPaperScissorsAgent.register(runtime, "rock_paper_scissors", lambda: RockPaperScissorsAgent("rock_paper_scissors"))
runtime.start()

In [ ]:
agent_id = AgentId("rock_paper_scissors", "default")
message = Message(content="go")
response = await runtime.send_message(message, agent_id)
print(response.content)

In [25]:
await runtime.stop()
await runtime.close()